# 🚦 Sidra Intersection – Worst Movement Summary Extractor

This notebook extracts key performance data from **SIDRA INTERSECTION** PDF reports and produces a clean Excel summary focused on the **worst performing movement** at each junction.

---

### 📋 Output Columns

| Column | Description |
|--------|-------------|
| Scenario | Time period (e.g. Base AM, Base PM) |
| Location | Junction reference (e.g. Jn 1, Jn 2) |
| V/C | Volume-to-Capacity ratio of the worst movement |
| Delay | Average delay (sec) of the worst movement |
| LOS | Level of Service of the worst movement |
| Max Queue Length | 95th percentile back-of-queue length (m) |
| Approach | Approach direction of the worst movement |
| W_Movement | Worst movement description (approach + turn) |
| W_Delay | Worst movement delay (sec) |
| W_V/C | Worst movement V/C ratio |
| W_LOS | Worst movement Level of Service |

---

### 🔢 LOS Thresholds (HCM)

| LOS | Delay (sec/veh) |
|-----|-----------------|
| A | ≤ 10 |
| B | 10 – 15 |
| C | 15 – 25 |
| D | 25 – 35 |
| E | 35 – 50 |
| F | > 50 |

---

### 📊 Sample Output

Below is an example of the kind of result this notebook produces:

| Scenario | Location | V/C | Delay | LOS | Max Queue Length | Approach | W_Movement | W_Delay | W_V/C | W_LOS |
|----------|----------|-----|-------|-----|-----------------|----------|------------|---------|-------|-------|
| Base AM | Jn 1 | 2.096 | 616.4 | F | 460.8 | SouthEast | SouthEast L2 | 616.4 | 2.096 | F |
| Base AM | Jn 2 | 4.130 | 1619.6 | F | 416.3 | NorthWest | NorthWest L2 | 1619.6 | 4.130 | F |
| Base AM | Jn 5 | 0.351 | 10.4 | B | 10.3 | SouthWest | SouthWest R2 | 10.4 | 0.351 | B |
| Base PM | Jn 1 | 3.241 | 1112.6 | F | 599.2 | SouthEast | SouthEast L2 | 1112.6 | 3.241 | F |

> **Note:** The worst movement is selected by the highest V/C ratio. When multiple movements share the same V/C, the one with the highest delay is chosen as the tie-breaker. Junctions are always sorted numerically (Jn 1 → Jn 2 → … → Jn 10 → Jn 11) regardless of any suffix or prefix in the name.

---

## Step 1 – Install Required Libraries

Run this cell **once** per Colab session. It installs:
- **pdfplumber** – extracts text from PDF pages
- **pymupdf** – splits multi-page PDFs into individual pages

In [ ]:
!pip install pdfplumber pymupdf --quiet
print('✅ Libraries installed.')

## Step 2 – Upload Sidra PDF Report(s)

A file picker will appear. Select **one or more** Sidra PDF reports to process.

> You can upload multiple PDFs at once (e.g. Base AM and Base PM reports together).

In [ ]:
from google.colab import files

print('Please select your Sidra PDF file(s) to upload...')
uploaded = files.upload()

print(f'\n✅ {len(uploaded)} file(s) uploaded successfully:')
for name in uploaded.keys():
    print(f'   • {name}')

## Step 3 – Import Libraries & Define Helper Functions

Imports all required libraries and defines three helper functions:
- **`categorize_value(delay)`** – converts average delay (sec) to an HCM LOS letter (A–F)
- **`natural_sort_key(filename)`** – sorts filenames numerically so `Sidra_JN_2` comes before `Sidra_JN_10`
- **`junction_sort_key(location)`** – extracts the junction number from any Location string for correct numeric ordering in the final output, ignoring any prefix or suffix (e.g. `Jn 34v - RIRO` → `34`)

In [ ]:
import pandas as pd
import pdfplumber
import fitz          # pymupdf
import re
import os
import shutil
import warnings
import logging

# Suppress noisy warnings from PDF libraries
warnings.filterwarnings('ignore')
logging.getLogger('pdfminer').setLevel(logging.ERROR)


def categorize_value(delay: float) -> str:
    """
    Convert average delay (sec) to HCM Level of Service letter.
    Always calculated from delay so 'NA' entries in the report are handled correctly.
    """
    if delay <= 10:          return 'A'
    elif 10 < delay <= 15:   return 'B'
    elif 15 < delay <= 25:   return 'C'
    elif 25 < delay <= 35:   return 'D'
    elif 35 < delay <= 50:   return 'E'
    else:                    return 'F'


def natural_sort_key(filename: str) -> list:
    """
    Natural sort: extracts all integers from a filename so that
    Sidra_JN_2.pdf sorts before Sidra_JN_10.pdf.
    """
    return [int(n) for n in re.findall(r'\d+', filename)]


def junction_sort_key(location: str) -> int:
    """
    Extracts the leading junction number from any Location string.
    Works with all formats:  'Jn 4 - RIRO' -> 4
                             'Jn 34v'       -> 34
                             'Junction 10'  -> 10
    """
    m = re.search(r'(\d+)', str(location))
    return int(m.group(1)) if m else 0


print('✅ Libraries imported and helper functions defined.')

## Step 4 – Split PDFs into Individual MOVEMENT SUMMARY Pages

Each Sidra PDF may contain multiple junctions. This step:
1. Scans every page of each uploaded PDF
2. Identifies pages that start with a **MOVEMENT SUMMARY** header
3. Saves each such page as a separate single-page PDF in the `output_pages/` folder

Splitting pages first makes parsing each junction independently much more reliable.

In [ ]:
OUTPUT_FOLDER = 'output_pages'

# Clean up any previous run
if os.path.exists(OUTPUT_FOLDER):
    shutil.rmtree(OUTPUT_FOLDER)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

count = 0
for filename in uploaded.keys():
    doc = fitz.open(filename)
    pages_found = 0
    for i, page in enumerate(doc):
        text  = page.get_text('text')
        lines = text.splitlines()
        if lines and 'MOVEMENT SUMMARY' in lines[0]:
            new_doc = fitz.open()
            new_doc.insert_pdf(doc, from_page=i, to_page=i)
            out_path = os.path.join(OUTPUT_FOLDER, f'Sidra_JN_{count + 1}.pdf')
            new_doc.save(out_path)
            count       += 1
            pages_found += 1
    print(f'   📄 {filename}  →  {pages_found} MOVEMENT SUMMARY page(s) found')

print(f'\n✅ Total pages extracted: {count}')

## Step 5 – Parse Each Page and Extract Worst Movement Data

For every extracted page this step:
- Reads the **Scenario** and **Location** from the page header
  - Handles all junction name formats: `Jn 1`, `Jn 1v`, `Jn 34 - RIRO`, `Jn 34v`, `Junction 10`
- Parses every individual **movement row** (lines containing `All MCs`) from the raw text
- Extracts V/C, Delay, LOS and Queue (m) for each movement
- Identifies the **worst movement** as the one with the **highest V/C ratio**
- Uses **highest delay** as a tie-breaker when multiple movements share the same V/C

> **Sorting:** Results are ordered by Scenario (in the order they appear in the PDF) then by junction number numerically — Jn 1 → Jn 2 → Jn 3 … → Jn 10 → Jn 11, regardless of any suffix or prefix.

In [ ]:
APPROACH_RE = re.compile(
    r'^(North(?:East|West)?|South(?:East|West)?|East|West|North|South)\s*:',
    re.IGNORECASE
)

results = []

# ── Natural sort: Sidra_JN_2.pdf comes before Sidra_JN_10.pdf ────────────
pdf_files = sorted(
    [f for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.pdf')],
    key=natural_sort_key
)

for pdf_file in pdf_files:
    with pdfplumber.open(os.path.join(OUTPUT_FOLDER, pdf_file)) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue

            lines = [l.strip() for l in text.splitlines()]

            # ── 1. Extract Scenario and Location from the page header ─────
            # Uses \S+ (not \d+) to capture site IDs like '34v', '1v' etc.
            scenario = tmc_number = None
            for line in lines:
                m = re.search(
                    r'Site:\s*\S+\s*\[(.+?)\s*\(Site Folder:\s*(.+?)\)\]', line
                )
                if m:
                    tmc_number = m.group(1).strip()   # e.g. 'Jn 1', 'Jn 34v'
                    scenario   = m.group(2).strip()   # e.g. 'Base AM'
                    break
            if not scenario:
                continue

            # ── 2. Parse every movement row from the raw text lines ───────
            # Movement rows always contain 'All MCs' — we parse these directly
            # from text (not the extracted table) because pdfplumber returns
            # the Sidra table as a single-column structure.
            movements        = []
            current_approach = None

            for line in lines:

                # Detect approach direction header (e.g. 'SouthEast: RoadName')
                am = APPROACH_RE.match(line)
                if am:
                    current_approach = am.group(1).strip()
                    continue

                # Only parse movement rows — they all contain 'All MCs'
                if 'All MCs' not in line or not current_approach:
                    continue

                # Row format: ID  Turn  All  MCs  flow  HV  flow  HV  v/c  delay  [LOS X]  queue_veh  queue_m  ...
                parts = line.split()
                try:
                    # Find position of 'MCs' to anchor the parse
                    mc_idx = next(i for i, p in enumerate(parts) if p == 'MCs')
                    turn   = parts[mc_idx - 2]   # turn label: L2, T1, R2, U etc.

                    # Collect all numeric values that follow 'All MCs'
                    nums = []
                    for p in parts[mc_idx + 1:]:
                        try:
                            nums.append(float(p))
                        except ValueError:
                            pass

                    # Positions after 'All MCs': flow(0) HV(1) flow(2) HV(3) v/c(4) delay(5) queue_veh(6) queue_m(7)
                    if len(nums) < 6:
                        continue

                    vc_val    = nums[4]
                    delay_val = nums[5]
                    queue_m   = nums[7] if len(nums) > 7 else None

                    # LOS: take from text if present, otherwise derive from delay
                    los_match = re.search(r'LOS\s+([A-F])', line)
                    los_val   = los_match.group(1) if los_match else categorize_value(delay_val)

                    movements.append({
                        'Approach': current_approach,
                        'Turn':     turn,
                        'v/c':      vc_val,
                        'Delay':    delay_val,
                        'LOS':      los_val,
                        'Queue_m':  queue_m,
                    })

                except (StopIteration, IndexError):
                    continue

            if not movements:
                continue

            # ── 3. Identify worst movement: highest V/C, tie-break = highest delay
            df     = pd.DataFrame(movements)
            max_vc = df['v/c'].max()
            top    = df[df['v/c'] == max_vc]
            best   = top.loc[top['Delay'].idxmax()]

            results.append({
                'Scenario':         scenario,
                'Location':         tmc_number,
                'V/C':              round(best['v/c'],   3),
                'Delay':            round(best['Delay'], 1),
                'LOS':              categorize_value(best['Delay']),
                'Max Queue Length': best['Queue_m'],
                'Approach':         best['Approach'],
                'W_Movement':       f"{best['Approach']} {best['Turn']}",
                'W_Delay':          round(best['Delay'], 1),
                'W_V/C':            round(best['v/c'],   3),
                'W_LOS':            categorize_value(best['Delay']),
            })

# ── 4. Build DataFrame and sort by Scenario order then junction number ────
if results:
    Output = pd.DataFrame(results)

    Output['_jn_num']     = Output['Location'].apply(junction_sort_key)
    scenario_order        = Output['Scenario'].unique().tolist()
    Output['_scen_order'] = Output['Scenario'].apply(lambda s: scenario_order.index(s))

    Output = (
        Output
        .sort_values(['_scen_order', '_jn_num'])
        .drop(columns=['_jn_num', '_scen_order'])
        .reset_index(drop=True)
    )

    print(f'✅ Parsed {len(Output)} junction(s) successfully.')
    print(f'   Scenarios  : {Output["Scenario"].unique().tolist()}')
    print(f'   Order check (first 5): {Output["Location"].head().tolist()}\n')
    display(Output)
else:
    print('⚠️  No junctions parsed. Please check:')
    print('   • The PDF was exported directly from SIDRA (not a scanned image)')
    print('   • Each page starts with a MOVEMENT SUMMARY header')
    Output = pd.DataFrame()

## Step 6 – Export Results to Excel

Saves the summary table to **Sidra_Worst_Movement_Report.xlsx** and downloads it automatically.

In [ ]:
if Output.empty:
    print('⚠️  Nothing to export — Output is empty. Please re-run Step 5.')
else:
    OUTPUT_FILE = 'Sidra_Worst_Movement_Report.xlsx'
    Output.to_excel(OUTPUT_FILE, index=False)
    print(f'✅ Report saved as "{OUTPUT_FILE}"')
    print(f'   Rows    : {len(Output)}')
    print(f'   Columns : {list(Output.columns)}')
    files.download(OUTPUT_FILE)
    print('\n⬇️  Download started.')

---
## ✅ Done!

Your **Sidra_Worst_Movement_Report.xlsx** has been downloaded.

---

### 🔧 Troubleshooting

| Issue | Likely Cause | Fix |
|-------|-------------|-----|
| `Parsed 0 junctions` | PDF is a scanned image, not a text-based export | Re-export the PDF directly from SIDRA |
| Missing junctions | Site header format is unusual | Check the line containing `Site Folder:` on that page |
| Junction out of order | Location contains no number | Verify the Location value — it will sort to position 0 |
| Wrong V/C or delay | Non-standard Sidra column order | Check the raw PDF — column shifts can occur in some Sidra versions |

---

### 📁 Supported Junction Name Formats

| Format | Example |
|--------|---------|
| Standard | `Jn 1`, `Jn 10` |
| With suffix | `Jn 1v`, `Jn 34v` |
| With descriptor | `Jn 4 - RIRO`, `Jn 10 - RIRO` |
| Full word | `Junction 1`, `Junction 10` |
| Combined | `Jn 34v - RIRO` |